In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

train = pd.read_csv("preprocessed_train_final.csv")
test  = pd.read_csv("preprocessed_test_final.csv")


c:\Users\ilker\anaconda3\envs\torchgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ilker\AppData\Local\Temp\ipykernel_31076\2607170246.py:6: DtypeWarning: Columns (0: product_id) have mixed types. Specify dtype option on import or set low_memory=False.
  test  = pd.read_csv("preprocessed_test_final.csv")


In [2]:
test = test.dropna(subset="category").reset_index(drop=True)
train_df = train[train["sample_weight"] == 1].reset_index(drop=True)

In [3]:
#train_df = train.groupby('category_fixed').sample(n=10, replace=True).reset_index(drop=True)
#test_df = test.groupby('category').sample(n=10, replace=True).reset_index(drop=True)

In [4]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics import accuracy_score, f1_score

# --- required columns in your dataframes ---
TEXT_COL  = "text_for_model"
LABEL_COL = "category_fixed"  # train_df has this per your message

# --- models ---
EMB_MODEL = "Trendyol/TY-ecomm-embed-multilingual-base-v1.2.0"
# multilingual reranker (works reasonably for TR/EN). You can swap if you have a better reranker.
RERANK_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

# --- retrieval / rerank params ---
FAISS_M = 48               # HNSW connectivity (higher => better recall, more memory)
EF_CONSTRUCTION = 200      # index build quality
EF_SEARCH = 256            # search quality

K_RETRIEVE = 200           # get top-200 candidates from FAISS
K_RERANK = 50              # rerank top-50 with cross-encoder (precision step)

# how to "decide" label from reranked candidates
DECIDE_MODE = "agg_max"    # {"top1_neighbor", "agg_max", "agg_sumexp"}
TOPK_LIST = [1, 3, 5]      # evaluation @k

# batching
EMB_BATCH = 256            # adjust for GPU RAM (RTX 2060 6GB: 128-256 often ok)
RERANK_BATCH = 256         # cross-encoder pair batch size
QUERY_BATCH = 256          # how many queries to process per loop


In [5]:
train_df[train_df['sample_weight'] == 1].shape

(2798565, 36)

In [6]:
# bi-encoder embedding model (SentenceTransformer-style)
embedder = SentenceTransformer(EMB_MODEL, device="cuda", trust_remote_code=True)

# cross-encoder reranker
reranker = CrossEncoder(RERANK_MODEL, device="cuda")


In [ ]:
train_texts = train_df[TEXT_COL].fillna("").astype(str).to_numpy()
train_labels = train_df[LABEL_COL].astype(str).to_numpy()

# infer embedding dimension by encoding a tiny sample
tmp = embedder.encode(["test"], batch_size=1, convert_to_numpy=True, normalize_embeddings=True)
d = int(tmp.shape[1])
print("Embedding dim:", d)

# HNSW index (L2)
index = faiss.IndexHNSWFlat(d, FAISS_M)  # L2 metric by default
index.hnsw.efConstruction = EF_CONSTRUCTION
index.hnsw.efSearch = EF_SEARCH


# add embeddings in batches
for start in tqdm(range(0, len(train_texts), EMB_BATCH), desc="Indexing train embeddings"):
    end = min(start + EMB_BATCH, len(train_texts))
    emb = embedder.encode(
        train_texts[start:end],
        batch_size=EMB_BATCH,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float32)
    index.add(emb)

print("Index size:", index.ntotal)


Embedding dim: 768


Indexing train embeddings:   1%|          | 58/10932 [00:39<2:07:54,  1.42it/s]

In [ ]:
def aggregate_label_scores(cand_labels, cand_scores, mode="agg_max"):
    # cand_labels: (K,) array of strings
    # cand_scores: (K,) array of reranker scores (higher=better)
    # returns dict label->score
    out = {}
    if mode == "top1_neighbor":
        out[cand_labels[0]] = float(cand_scores[0])
        return out

    if mode == "agg_max":
        for lab, sc in zip(cand_labels, cand_scores):
            sc = float(sc)
            if lab not in out or sc > out[lab]:
                out[lab] = sc
        return out

    if mode == "agg_sumexp":
        # softer vote: sum(exp(score)) per label
        for lab, sc in zip(cand_labels, cand_scores):
            out[lab] = out.get(lab, 0.0) + float(np.exp(sc))
        return out

    raise ValueError("Unknown mode")

def predict_retrieve_rerank(texts: np.ndarray):
    N = len(texts)
    pred_top1 = np.empty(N, dtype=object)
    pred_topk = {k: [None]*N for k in TOPK_LIST}

    for q0 in tqdm(range(0, N, QUERY_BATCH), desc="Retrieve→Rerank"):
        q1 = min(q0 + QUERY_BATCH, N)
        q_texts = texts[q0:q1]

        # embed queries
        q_emb = embedder.encode(
            q_texts,
            batch_size=EMB_BATCH,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        ).astype(np.float32)

        # retrieve candidates
        D, I = index.search(q_emb, K_RETRIEVE)  # I: (b, K_RETRIEVE)

        # rerank only top K_RERANK
        I_top = I[:, :K_RERANK]
        bsz = I_top.shape[0]

        # build pair list for cross-encoder: (query, candidate_text)
        pair_q = np.repeat(q_texts, K_RERANK)
        pair_c = train_texts[I_top.reshape(-1)]
        pairs = list(zip(pair_q.tolist(), pair_c.tolist()))

        # score pairs in batches
        scores = []
        for p0 in range(0, len(pairs), RERANK_BATCH):
            p1 = min(p0 + RERANK_BATCH, len(pairs))
            s = reranker.predict(pairs[p0:p1])
            scores.append(np.asarray(s, dtype=np.float32))
        scores = np.concatenate(scores, axis=0).reshape(bsz, K_RERANK)

        # decide label per query
        for i in range(bsz):
            cand_idx = I_top[i]
            cand_labs = train_labels[cand_idx]
            cand_sco = scores[i]

            # sort candidates by reranker score desc
            order = np.argsort(-cand_sco)
            cand_labs = cand_labs[order]
            cand_sco  = cand_sco[order]

            lab2score = aggregate_label_scores(cand_labs, cand_sco, mode=DECIDE_MODE)
            # ranked labels
            ranked = sorted(lab2score.items(), key=lambda x: -x[1])
            ranked_labels = [x[0] for x in ranked]

            pred_top1[q0+i] = ranked_labels[0] if ranked_labels else cand_labs[0]

            for k in TOPK_LIST:
                pred_topk[k][q0+i] = ranked_labels[:k] if len(ranked_labels) >= k else ranked_labels

    return pred_top1, pred_topk

# run on test
test_texts = test_df[TEXT_COL].fillna("").astype(str).to_numpy()
pred_top1, pred_topk = predict_retrieve_rerank(test_texts)

test_df["pred_rac_top1"] = pred_top1
for k in TOPK_LIST:
    test_df[f"pred_rac_top{k}"] = pred_topk[k]

test_df[["pred_rac_top1"] + [f"pred_rac_top{k}" for k in TOPK_LIST]].head()


Retrieve→Rerank: 100%|██████████| 45/45 [08:15<00:00, 11.00s/it]


,pred_rac_top1,pred_rac_top1,pred_rac_top3,pred_rac_top5
0,[ADSL Modemler],[ADSL Modemler],"[ADSL Modemler, Ölçüm Cihazları, Güvenlik Ciha...","[ADSL Modemler, Ölçüm Cihazları, Güvenlik Ciha..."
1,[Elektronik Devre Elemanları],[Elektronik Devre Elemanları],"[Elektronik Devre Elemanları, ADSL Modemler, A...","[Elektronik Devre Elemanları, ADSL Modemler, A..."
2,[ADSL Modemler],[ADSL Modemler],"[ADSL Modemler, Çelik Para Kasası, Güvenlik Ci...","[ADSL Modemler, Çelik Para Kasası, Güvenlik Ci..."
3,[ADSL Modemler],[ADSL Modemler],"[ADSL Modemler, Oto Hoparlör, SSD (Solid State...","[ADSL Modemler, Oto Hoparlör, SSD (Solid State..."
4,[ADSL Modemler],[ADSL Modemler],"[ADSL Modemler, Elektronik Devre Elemanları, S...","[ADSL Modemler, Elektronik Devre Elemanları, S..."


In [ ]:
train_freq = train_df[LABEL_COL].astype(str).value_counts()
freq_tbl = train_freq.rename("train_cnt").reset_index().rename(columns={"index":"category_fixed"})
freq_tbl["cum_cnt"] = freq_tbl["train_cnt"].cumsum()
freq_tbl["cum_share"] = freq_tbl["cum_cnt"] / freq_tbl["train_cnt"].sum()

def to_hbt(cum_share):
    if cum_share <= 0.80:
        return "head"
    elif cum_share <= 0.95:
        return "body"
    else:
        return "tail"

freq_tbl["hbt"] = freq_tbl["cum_share"].map(to_hbt)
hbt_map = dict(zip(freq_tbl["category_fixed"], freq_tbl["hbt"]))

freq_tbl.groupby("hbt").agg(n_labels=("category_fixed","count"), train_rows=("train_cnt","sum"))


,n_labels,train_rows
hbt,,
body,171,1710
head,912,9120
tail,57,570


In [ ]:
import ast

def unwrap_top1(x):
    # if it's a python list -> take first
    if isinstance(x, (list, tuple)) and len(x) > 0:
        return x[0]
    # if it's a string representation like "['ADSL Modemler']"
    if isinstance(x, str) and x.startswith("[") and x.endswith("]"):
        try:
            v = ast.literal_eval(x)
            if isinstance(v, list) and len(v) > 0:
                return v[0]
        except Exception:
            pass
    return x

test_df["pred_rac_top1"] = test_df["pred_rac_top1"].apply(unwrap_top1)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_true = test_df["category"].astype(str).to_numpy()
y_pred = test_df["pred_rac_top1"].astype(str).to_numpy()

# hbt assignment based on TRUE label
hbt = pd.Series(y_true).map(hbt_map).fillna("tail").to_numpy()

def metrics_block(mask):
    yt = y_true[mask]
    yp = y_pred[mask]
    out = {
        "n_rows": int(mask.sum()),
        "n_labels": int(pd.Series(yt).nunique()),
        "acc": float(accuracy_score(yt, yp)),
        "macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(yt, yp, average="weighted", zero_division=0)),
    }

    # top-k accuracy (true label in predicted list)
    for k in [3, 5]:
        col = f"pred_rac_top{k}"
        if col in test_df.columns:
            topk_lists = test_df.loc[mask, col].values
            ok = np.mean([yt_i in topk_i for yt_i, topk_i in zip(yt, topk_lists)])
            out[f"top{k}_acc"] = float(ok)

    return out

rows = []
for part in ["head", "body", "tail"]:
    mask = (hbt == part)
    r = metrics_block(mask)
    r["slice"] = part
    rows.append(r)

mask_all = np.ones(len(test_df), dtype=bool)
r = metrics_block(mask_all)
r["slice"] = "ALL"
rows.append(r)

hbt_metrics = pd.DataFrame(rows)
score_cols = [c for c in hbt_metrics.columns if c not in ["slice","n_rows","n_labels"]]
hbt_metrics[score_cols] = hbt_metrics[score_cols].round(4)
hbt_metrics[["slice","n_rows","n_labels"] + score_cols]


,slice,n_rows,n_labels,acc,macro_f1,weighted_f1,top3_acc,top5_acc
0,head,9120,912,0.5309,0.4359,0.5348,0.7480,0.8309
1,body,1710,171,0.5298,0.2007,0.6184,0.7596,0.8485
2,tail,570,57,0.5088,0.1553,0.6129,0.7544,0.8474
3,ALL,11400,1140,0.5296,0.5149,0.5149,0.7501,0.8344


In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import pandas as pd

y_true = test_df["category"].astype(str).values
y_pred = test_df["pred_rac_top1"].astype(str).values  # must be plain strings

# HBT slice from TRUE label
hbt = pd.Series(y_true).map(hbt_map).fillna("tail").values

def row_metrics(mask, name):
    yt = y_true[mask]
    yp = y_pred[mask]
    return {
        "slice": name,
        "n": int(mask.sum()),
        "acc": accuracy_score(yt, yp),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "n_labels": pd.Series(yt).nunique()
    }

rows = []
for s in ["head","body","tail"]:
    rows.append(row_metrics(hbt == s, s))
rows.append(row_metrics(np.ones(len(y_true), dtype=bool), "ALL"))

pd.DataFrame(rows).round(4)


,slice,n,acc,micro_f1,macro_f1,weighted_f1,n_labels
0,head,9120,0.5309,0.5309,0.4359,0.5348,912
1,body,1710,0.5298,0.5298,0.2007,0.6184,171
2,tail,570,0.5088,0.5088,0.1553,0.6129,57
3,ALL,11400,0.5296,0.5296,0.5149,0.5149,1140


In [ ]:
import numpy as np
import pandas as pd
import ast
from sklearn.metrics import accuracy_score, f1_score

LABEL_COL = "category_fixed"   # true labels in test_df
PRED1_COL = "pred_rac_top1"    # your top-1 string predictions
TOPK_COLS = ["pred_rac_top3", "pred_rac_top5"]  # optional if you have them

# --- sanity: top1 must be a plain string, not "['x']"
def unwrap_top1(x):
    if isinstance(x, (list, tuple)) and len(x) > 0:
        return x[0]
    if isinstance(x, str) and x.startswith("[") and x.endswith("]"):
        try:
            v = ast.literal_eval(x)
            if isinstance(v, list) and len(v) > 0:
                return v[0]
        except Exception:
            pass
    return x

test_df[PRED1_COL] = test_df[PRED1_COL].apply(unwrap_top1).astype(str)

# --- helper for top-k lists (if stored as string lists)
def ensure_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, str) and x.startswith("[") and x.endswith("]"):
        try:
            v = ast.literal_eval(x)
            return v if isinstance(v, list) else [str(v)]
        except Exception:
            return [x]
    return [str(x)]

# --- build head/body/tail sets from TRAIN frequency (you already do this, but keeping it self-contained)
train_freq = train_df[LABEL_COL].astype(str).value_counts()
freq_tbl = train_freq.rename("train_cnt").reset_index().rename(columns={"index":"category_fixed"})
freq_tbl["cum_cnt"] = freq_tbl["train_cnt"].cumsum()
freq_tbl["cum_share"] = freq_tbl["cum_cnt"] / freq_tbl["train_cnt"].sum()

def to_hbt(c):
    if c <= 0.80:
        return "head"
    elif c <= 0.95:
        return "body"
    else:
        return "tail"

freq_tbl["hbt"] = freq_tbl["cum_share"].map(to_hbt)
hbt_map = dict(zip(freq_tbl["category_fixed"], freq_tbl["hbt"]))

# --- data arrays
y_true_all = test_df["category"].astype(str).values
y_pred_all = test_df[PRED1_COL].astype(str).values
hbt_all = pd.Series(y_true_all).map(hbt_map).fillna("tail").values

ALL_LABELS = np.unique(y_true_all)  # global label set based on truths

rows = []

for sl in ["head", "body", "tail", "ALL"]:
    if sl == "ALL":
        mask = np.ones(len(test_df), dtype=bool)
    else:
        mask = (hbt_all == sl)

    yt = y_true_all[mask]
    yp = y_pred_all[mask]

    # slice label set (THIS is the fix!)
    slice_labels = np.unique(yt)

    # diagnostics: how often does the model predict labels outside this slice’s true label set?
    oos_rate = float(np.mean(~np.isin(yp, slice_labels))) if len(yp) else 0.0

    out = {
        "slice": sl,
        "n_rows": int(mask.sum()),
        "n_true_labels_in_slice": int(len(slice_labels)),
        "acc": float(accuracy_score(yt, yp)) if len(yt) else 0.0,
        "micro_f1": float(f1_score(yt, yp, average="micro", zero_division=0)) if len(yt) else 0.0,
        "weighted_f1": float(f1_score(yt, yp, average="weighted", zero_division=0)) if len(yt) else 0.0,

        # ✅ FIXED macro: only average over labels that exist in yt for that slice
        "macro_f1_sliceLabels": float(f1_score(yt, yp, average="macro", labels=slice_labels, zero_division=0)) if len(yt) else 0.0,

        # optional: global-macro within slice (usually very low, keep for honesty if you want)
        "macro_f1_globalLabels": float(f1_score(yt, yp, average="macro", labels=ALL_LABELS, zero_division=0)) if len(yt) else 0.0,

        "pred_out_of_slice_rate": oos_rate
    }

    # Top-k accuracy
    for col in TOPK_COLS:
        if col in test_df.columns:
            topk_lists = test_df.loc[mask, col].apply(ensure_list).values
            hit = np.mean([t in preds for t, preds in zip(yt, topk_lists)]) if len(yt) else 0.0
            out[f"top{col[-1]}_acc"] = float(hit)  # pred_rac_top3 -> top3_acc, pred_rac_top5 -> top5_acc

    rows.append(out)

hbt_metrics_fixed = pd.DataFrame(rows)

# nice rounding
num_cols = [c for c in hbt_metrics_fixed.columns if c not in ["slice"]]
hbt_metrics_fixed[num_cols] = hbt_metrics_fixed[num_cols].astype(float).round(4)

hbt_metrics_fixed


,slice,n_rows,n_true_labels_in_slice,acc,micro_f1,weighted_f1,macro_f1_sliceLabels,macro_f1_globalLabels,pred_out_of_slice_rate,top3_acc,top5_acc
0,head,9120.0,912.0,0.5309,0.5309,0.5348,0.5348,0.4278,0.0839,0.7480,0.8309
1,body,1710.0,171.0,0.5298,0.5298,0.6184,0.6184,0.0928,0.3515,0.7596,0.8485
2,tail,570.0,57.0,0.5088,0.5088,0.6129,0.6129,0.0306,0.4281,0.7544,0.8474
3,ALL,11400.0,1140.0,0.5296,0.5296,0.5149,0.5149,0.5149,0.0000,0.7501,0.8344
